# AI-Based Gardening Video Recommendation System

## Project Overview

This project creates a content-based recommendation system that recommends
gardening video topics based on a user's gardening interests.

The system considers the plant or garden area, gardening topic, experience
level, and an optional user question. It uses TF-IDF and cosine similarity
to identify gardening video topics that most closely match the user's needs.

The project adapts recommendation-system concepts from the movie
recommendation examples provided in the course and applies them to
home gardening.

### Technologies Used
- Python
- Pandas
- Scikit-learn
- TF-IDF
- Cosine Similarity
- Gradio

In [ ]:
# Import libraries used for data processing and recommendations

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Load the gardening video dataset

videos = pd.read_csv("/content/gardening_videos.csv")

print("Dataset loaded successfully!")
print("Number of gardening topics:", len(videos))

Dataset loaded successfully!
Number of gardening topics: 74


In [ ]:
# Display the first 10 gardening topics

videos.head(10)

,title,plant,category,topic,level,keywords
0,Growing Cucumbers Vertically on a Cattle Panel...,Cucumber,Vining Vegetables,Trellising,Beginner,cucumber vertical gardening cattle panel trell...
1,Pruning Cucumber Vines for Better Airflow,Cucumber,Vining Vegetables,Pruning,Intermediate,cucumber pruning vines airflow disease prevent...
2,Cucumber Watering Guide for Raised Beds,Cucumber,Vining Vegetables,Watering,Beginner,cucumber watering raised bed moisture mulch su...
3,Common Cucumber Pests and Organic Control,Cucumber,Vining Vegetables,Pest Control,Beginner,cucumber beetle aphids pests organic control neem
4,Why Cucumber Leaves Turn Yellow,Cucumber,Vining Vegetables,Disease,Beginner,cucumber yellow leaves nutrient deficiency ove...
5,Training Pole Beans on a Trellis,String Bean,Vining Vegetables,Trellising,Beginner,pole beans string beans trellis climbing vines...
6,Growing String Beans in Raised Beds,String Bean,Vining Vegetables,Planting,Beginner,string beans pole beans raised bed planting sp...
7,Harvesting String Beans for Continued Production,String Bean,Vining Vegetables,Harvesting,Beginner,string beans harvest picking production tender...
8,Squash Trellising and Vine Support,Squash,Vining Vegetables,Trellising,Intermediate,squash trellis vertical support vines fruit sl...
9,Squash Vine Borer Prevention and Control,Squash,Vining Vegetables,Pest Control,Intermediate,squash vine borer pest prevention organic control


In [ ]:
# Check the dataset structure

videos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74 entries, 0 to 73
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     74 non-null     object
 1   plant     74 non-null     object
 2   category  74 non-null     object
 3   topic     74 non-null     object
 4   level     74 non-null     object
 5   keywords  74 non-null     object
dtypes: object(6)
memory usage: 3.6+ KB


In [ ]:
# Check for missing values

videos.isnull().sum()

,0
title,0
plant,0
category,0
topic,0
level,0
keywords,0


In [ ]:
# Combine descriptive fields into one text feature

videos["combined_text"] = (
    videos["title"] + " " +
    videos["plant"] + " " +
    videos["category"] + " " +
    videos["topic"] + " " +
    videos["level"] + " " +
    videos["keywords"]
).str.lower()

videos[["title", "combined_text"]].head()

,title,combined_text
0,Growing Cucumbers Vertically on a Cattle Panel...,growing cucumbers vertically on a cattle panel...
1,Pruning Cucumber Vines for Better Airflow,pruning cucumber vines for better airflow cucu...
2,Cucumber Watering Guide for Raised Beds,cucumber watering guide for raised beds cucumb...
3,Common Cucumber Pests and Organic Control,common cucumber pests and organic control cucu...
4,Why Cucumber Leaves Turn Yellow,why cucumber leaves turn yellow cucumber vinin...


In [ ]:
# Convert gardening text into numerical TF-IDF features

tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(
    videos["combined_text"]
)

print("TF-IDF matrix created successfully!")
print("Matrix shape:", tfidf_matrix.shape)

TF-IDF matrix created successfully!
Matrix shape: (74, 963)


In [ ]:
# Calculate similarity between all gardening topics

similarity_matrix = cosine_similarity(
    tfidf_matrix
)

print("Similarity matrix created successfully!")
print("Matrix shape:", similarity_matrix.shape)

Similarity matrix created successfully!
Matrix shape: (74, 74)


In [ ]:
# Create the gardening recommendation function

def recommend_gardening_videos(plant, topic, level, user_interest, number=5):

    # Combine the user's selections into one search query
    query_parts = []

    if plant and plant != "Any":
        query_parts.append(plant)

    if topic and topic != "Any":
        query_parts.append(topic)

    if level and level != "Any":
        query_parts.append(level)

    if user_interest:
        query_parts.append(user_interest)

    # Check if the user entered any information
    if len(query_parts) == 0:
        return "Please select or enter at least one gardening preference."

    # Combine the user's choices
    query_text = " ".join(query_parts).lower()

    # Convert the user's request into TF-IDF features
    query_vector = tfidf.transform([query_text])

    # Compare the request with all gardening topics
    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Rank the gardening topics from most similar to least similar
    ranked_indices = scores.argsort()[::-1]

    # Keep only results with a similarity score greater than zero
    recommendations = [
        index for index in ranked_indices
        if scores[index] > 0
    ][:int(number)]

    # Create the recommendation results
    results = []

    for rank, index in enumerate(recommendations, start=1):

        row = videos.iloc[index]

        results.append({
            "Rank": rank,
            "Recommended Video Topic": row["title"],
            "Plant": row["plant"],
            "Topic": row["topic"],
            "Level": row["level"],
            "Similarity": round(scores[index] * 100, 1)
        })

    return pd.DataFrame(results)

In [ ]:
# Test 1: Cucumber trellising recommendation

recommend_gardening_videos(
    plant="Cucumber",
    topic="Trellising",
    level="Beginner",
    user_interest="cattle panel vertical gardening",
    number=5
)

,Rank,Recommended Video Topic,Plant,Topic,Level,Similarity
0,1,Growing Cucumbers Vertically on a Cattle Panel...,Cucumber,Trellising,Beginner,55.9
1,2,Using a Cattle Panel as a Garden Arch Trellis,General,Trellising,Beginner,42.1
2,3,Squash Trellising and Vine Support,Squash,Trellising,Intermediate,24.8
3,4,Cucumber Watering Guide for Raised Beds,Cucumber,Watering,Beginner,12.0
4,5,Why Cucumber Leaves Turn Yellow,Cucumber,Disease,Beginner,11.9


In [ ]:
# Test 2: Tomato pest-control recommendation

recommend_gardening_videos(
    plant="Tomato",
    topic="Pest Control",
    level="Beginner",
    user_interest="hornworms and aphids",
    number=5
)

,Rank,Recommended Video Topic,Plant,Topic,Level,Similarity
0,1,Managing Aphids on Flowering Plants,Flowering Plants,Pest Control,Beginner,50.8
1,2,Tomato Hornworm and Aphid Control,Tomato,Pest Control,Beginner,47.6
2,3,Common Cucumber Pests and Organic Control,Cucumber,Pest Control,Beginner,34.2
3,4,Organic Pest Management for a Home Garden,General,Pest Control,Beginner,27.5
4,5,Squash Vine Borer Prevention and Control,Squash,Pest Control,Intermediate,23.8


In [ ]:
# Test 3: Container citrus recommendation

recommend_gardening_videos(
    plant="Citrus",
    topic="Fertilizing",
    level="Intermediate",
    user_interest="potted lemon and orange trees",
    number=5
)

,Rank,Recommended Video Topic,Plant,Topic,Level,Similarity
0,1,Citrus Fertilizing Schedule for Container Trees,Citrus,Fertilizing,Intermediate,57.9
1,2,Growing Orange Trees in Pots,Orange,Container Growing,Beginner,34.8
2,3,Growing Lemon Trees in Containers,Lemon,Container Growing,Beginner,27.1
3,4,Pruning Container Citrus Trees,Citrus,Pruning,Beginner,18.1
4,5,Fertilizing Pepper Plants for More Fruit,Bell Pepper,Fertilizing,Intermediate,16.3


In [ ]:
# Test 4: Blank input validation

recommend_gardening_videos(
    plant="Any",
    topic="Any",
    level="Any",
    user_interest="",
    number=5
)

'Please select or enter at least one gardening preference.'

## Improving the Recommendation Algorithm

Initial testing showed a limitation in the first version of the recommendation
system. For a search involving Tomato, Pest Control, Beginner, and
"hornworms and aphids," a general flowering-plant aphid topic ranked higher
than the more relevant tomato-specific result.

This occurred because TF-IDF and cosine similarity primarily measure textual
similarity and did not give additional importance to the user's selected
plant, topic, or experience level.

To improve the recommendations, the ranking function was modified to add
extra weight for exact matches:

- Plant match: +0.20
- Topic match: +0.15
- Experience-level match: +0.05

The updated system was then retested using the same input to determine
whether the modification improved the ranking.

In [ ]:
# Improved gardening recommendation function
# Adds extra weight for exact plant, topic, and experience-level matches

def recommend_gardening_videos(plant, topic, level, user_interest, number=5):

    query_parts = []

    if plant and plant != "Any":
        query_parts.append(plant)

    if topic and topic != "Any":
        query_parts.append(topic)

    if level and level != "Any":
        query_parts.append(level)

    if user_interest:
        query_parts.append(user_interest)

    # Validate the user's input
    if len(query_parts) == 0:
        return "Please select or enter at least one gardening preference."

    # Combine user selections into one search query
    query_text = " ".join(query_parts).lower()

    # Convert the user's request into TF-IDF features
    query_vector = tfidf.transform([query_text])

    # Calculate text similarity
    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Add extra weight for exact category matches
    for index, row in videos.iterrows():

        if plant != "Any" and row["plant"] == plant:
            scores[index] += 0.20

        if topic != "Any" and row["topic"] == topic:
            scores[index] += 0.15

        if level != "Any" and row["level"] == level:
            scores[index] += 0.05

    # Rank recommendations
    ranked_indices = scores.argsort()[::-1]

    recommendations = [
        index for index in ranked_indices
        if scores[index] > 0
    ][:int(number)]

    # Prepare results
    results = []

    for rank, index in enumerate(recommendations, start=1):

        row = videos.iloc[index]

        results.append({
            "Rank": rank,
            "Recommended Video Topic": row["title"],
            "Plant": row["plant"],
            "Topic": row["topic"],
            "Level": row["level"],
            "Match Score": round(scores[index] * 100, 1)
        })

    return pd.DataFrame(results)

In [ ]:
# Re-test the same tomato pest-control request after improving the algorithm

recommend_gardening_videos(
    plant="Tomato",
    topic="Pest Control",
    level="Beginner",
    user_interest="hornworms and aphids",
    number=5
)

,Rank,Recommended Video Topic,Plant,Topic,Level,Match Score
0,1,Tomato Hornworm and Aphid Control,Tomato,Pest Control,Beginner,87.6
1,2,Managing Aphids on Flowering Plants,Flowering Plants,Pest Control,Beginner,70.8
2,3,Common Cucumber Pests and Organic Control,Cucumber,Pest Control,Beginner,54.2
3,4,Organic Pest Management for a Home Garden,General,Pest Control,Beginner,47.5
4,5,How to Prune Tomato Suckers,Tomato,Pruning,Beginner,45.6


## Graphical User Interface

After testing and improving the recommendation algorithm, a graphical user
interface was created using Gradio. The interface allows users to interact
with the recommendation system without writing Python code.

Users can select a plant or garden area, choose the type of gardening help
needed, select an experience level, enter an optional gardening question,
and choose the number of recommendations they want to receive.

In [ ]:
# Import Gradio for the graphical user interface

import gradio as gr

In [ ]:
# Create dropdown options from the gardening dataset

plant_options = ["Any"] + sorted(videos["plant"].unique().tolist())

topic_options = ["Any"] + sorted(videos["topic"].unique().tolist())

level_options = ["Any", "Beginner", "Intermediate", "Advanced"]

print("Plant options:", len(plant_options))
print("Topic options:", len(topic_options))
print("Interface options created successfully!")

Plant options: 30
Topic options: 13
Interface options created successfully!


In [ ]:
# Create the Gradio interface for the recommendation system

with gr.Blocks(title="AI Gardening Video Recommendation System") as demo:

    gr.Markdown("""
    # 🌱 AI-Based Gardening Video Recommendation System

    Select what you are growing and what you need help with.
    The system will recommend gardening video topics based on your interests.
    """)

    with gr.Row():

        plant_input = gr.Dropdown(
            choices=plant_options,
            value="Any",
            label="Plant or Garden Area"
        )

        topic_input = gr.Dropdown(
            choices=topic_options,
            value="Any",
            label="What Do You Need Help With?"
        )

        level_input = gr.Dropdown(
            choices=level_options,
            value="Any",
            label="Experience Level"
        )

    interest_input = gr.Textbox(
        label="Describe Your Gardening Question (Optional)",
        placeholder="Example: hornworms and aphids on my tomato plants"
    )

    number_input = gr.Slider(
        minimum=3,
        maximum=10,
        value=5,
        step=1,
        label="Number of Recommendations"
    )

    recommend_button = gr.Button(
        "Recommend Gardening Videos",
        variant="primary"
    )

    recommendation_output = gr.Dataframe(
        label="Recommended Gardening Video Topics"
    )

    recommend_button.click(
        fn=recommend_gardening_videos,
        inputs=[
            plant_input,
            topic_input,
            level_input,
            interest_input,
            number_input
        ],
        outputs=recommendation_output
    )

In [ ]:
# Launch the final recommendation system

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6f6f998a31d5d21c19.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
